# CIC heatmap (1970 vs 2022)


In [ ]:
# Load dependencies and make src/ importable from the notebook.
# Merge CIC data with commune/arrondissement geometries.
from pathlib import Path
import sys
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

root = Path.cwd()
sys.path.insert(0, str(root.parent / "src"))
    
from utils_cic_geo_merge import *

gdf = merge_cic_communes_then_arrondissements()


In [2]:
# Check missing values per column.
gdf.isna().sum().sort_values(ascending=False)

code_insee     2131216
code              2414
geometry           124
dep                  0
nomdep               0
codecommune          0
nomcommune           0
plm                  0
paris                0
value                0
metric               0
year                 0
dtype: int64

In [3]:
# List commune names that still have missing geometry.
gdf[gdf["geometry"].isna()]["nomcommune"].unique()

<StringArray>
['SANNERVILLE', 'BERNIERES-SUR-SEINE']
Length: 2, dtype: string

In [4]:
# Drop rows without geometry before spatial operations.
gdf = gdf[~gdf["geometry"].isna()]

In [ ]:
# Compute total pairwise overlap area between unique commune geometries.
gdf_unique = gdf.groupby("codecommune").first().reset_index(drop=True)
g = gdf_unique.reset_index(drop=True)[["geometry"]].copy()

pairs = gpd.sjoin(g, g, predicate="intersects", how="inner")
pairs = pairs[pairs.index < pairs.index_right]

geom_left = g.loc[pairs.index, "geometry"].to_numpy()
geom_right = g.loc[pairs.index_right, "geometry"].to_numpy()

intersect_area = gpd.GeoSeries(geom_left).intersection(gpd.GeoSeries(geom_right)).area.sum()


In [67]:
# Compare overlap area to the total covered area.
tot_area = gdf_unique.geometry.union_all().area

print(np.round(intersect_area/tot_area,4))

0.0007


In [ ]:
# Plot heatmaps for two reference years.
years = ["1970", "2022"]
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

for ax, year in zip(axes, years):
    subset = gdf[gdf["year"] == year]
    subset.plot(
        column="value",
        ax=ax,
        legend=True,
        cmap="viridis",
        missing_kwds={"color": "lightgrey", "label": "Missing"},
    )
    ax.set_title(f"CIC value in {year}")
    ax.set_axis_off()

plt.tight_layout()
